# Build temporal dataset from raw RFID logs

## Goal
Produce a dataset where each row is one **antRound** (one full Tx sweep) at one position,
preserving the full RSSI evolution over time — instead of collapsing to a single max/min per position.

## What is in each raw `.txt` file
Each file is a stream of JSON events at a fixed position and power level.
- `RoundStart`: configures active Tx antenna and its Rx pair (`rxAntennaConfig`)
- `TagReadData`: one RFID tag backscatter observation with **two RSSI values**:
  - `rssi[0]` = rssi1 at `rx1` (primary Rx antenna)
  - `rssi[1]` = rssi2 at `rx2` (secondary Rx antenna)
- Both are real measurements in `pgm=AEP_OK` (dual-receiver mode).
  In `pgm=AEP`, rssi2 is always -100 (single receiver) — so we only load `AEP_OK`.

## Antenna geometry
- **8 physical Tx antennas** at known (x,y,z) positions
- **4 internal Rx ports** (PORT_1..4) that also double as Tx
- **P=24**: each Tx has one fixed (rx1, rx2) pair → simple sweep
- **P=27/31.5**: each Tx cycles through all (rx1, rx2) combinations → **full bistatic MIMO scan**
- After "melt" (unpivoting rssi1/rssi2), features are indexed by **(Tx, Rx) pair** — up to 8×4 = 32 unique pairs

## Dataset representations (raw → models)

### A. Long events (intermediate — built in this notebook)
One row = one RSSI observation: `(file, antRound, Tx, Rx, rssi_dBm)`. Most flexible, good for exploratory analysis.

### B. Wide temporal (main output)
One row = one antRound. Columns are `{Tx}_{Rx}_max`, `{Tx}_{Rx}_min`, `{Tx}_{Rx}_n`.
- **Tabular models** (ExtraTrees, XGBoost, MLP): use each row independently
- **Sequence models** (LSTM, 1D-CNN, Transformer): group by `file_id` → `(T, F)` tensor with padding/mask
- **Matrix models** (GNN, attention over antenna pairs): reshape each antRound's features into a `(n_Tx, n_Rx)` RSSI matrix

### C. Aggregated (legacy-compatible)
Collapse all antRounds per file back to a single row with richer stats (max, min, mean, std, percentiles).
Same structure as the current `data/*.csv` datasets but with more features.

## CV rule
Always `GroupKFold` on `position_id`. All antRounds from the same position are correlated.

In [ ]:
import json
import re
import glob
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

## 1. Configuration

In [ ]:
RAW_ROOT    = Path("../raw_data")
CONF_DIR    = RAW_ROOT / "ConfID=2026-02-03__site=HQ"
ACTUALS_CSV = CONF_DIR / "Actuals/site=HQ__actual=P.csv"
ANTCONF_CSV = CONF_DIR / "antConf/antConf=HQ.csv"
OUT_DIR     = Path("../data")
OUT_DIR.mkdir(exist_ok=True)

# Only pgm=AEP_OK has dual-RX (real rssi2); pgm=AEP has rssi2=-100.
INCLUDE_PGMS = {"pgm=AEP_OK"}

# Keep only the first N seconds of each recording (0 = keep all)
MAX_DURATION_S = 20

# Tx antenna ordering for antRound detection (cycle wrap detection).
# An antRound increments when Tx sequence number decreases.
TX_ORDER = {
    "041A9F60__NONE": 1,
    "041A9F61__NONE": 2,
    "PORT_1__NONE":   3,
    "PORT_2__NONE":   4,
    "PORT_3__NONE":   5,
    "PORT_4__NONE":   6,
    "0F173B12__NONE": 7,
    "0F173B13__NONE": 8,
}

## 2. Raw file parsing helpers

In [ ]:
def parse_raw_file(path: Path) -> tuple[list, list]:
    """Return (tag_events, round_start_events) parsed from a .txt RFID log."""
    content = path.read_text()
    tags, rounds = [], []
    for block in content.strip().split("\n\n"):
        lines = block.strip().split("\n")
        data_line = next((l for l in lines if l.startswith("data:")), None)
        if data_line is None:
            continue
        try:
            obj = json.loads(data_line[len("data: "):])
        except json.JSONDecodeError:
            continue
        t = obj.get("type")
        if t == "TagReadData":
            tags.append(obj)
        elif t == "RoundStart":
            rounds.append(obj)
    return tags, rounds


def parse_filename(path: Path) -> dict | None:
    """Extract x, y, z, freq_P from a position-encoded filename."""
    num = r"[0-9]+(?:\.[0-9]+)?"
    m = re.search(rf"x=({num})__y=({num})__z0?=({num})__P=({num})", path.name)
    if m is None:
        return None
    return {
        "x": float(m.group(1)),
        "y": float(m.group(2)),
        "z": float(m.group(3)),
        "freq_P": float(m.group(4)),
    }


def build_round_info(rounds: list) -> tuple[dict, dict, dict]:
    """From RoundStart events, build:
    - round_to_ar:  round_id → antRound_rel (0-based)
    - round_to_tx:  round_id → Tx key (e.g. "PORT_1__NONE")
    - round_to_rx:  round_id → (rx1_key, rx2_key)
    """
    round_to_ar: dict[int, int] = {}
    round_to_tx: dict[int, str] = {}
    round_to_rx: dict[int, tuple[str, str]] = {}

    antRound = 0
    prev_seq = None
    for r in sorted(rounds, key=lambda x: x["timestamp"]):
        tx_key = r["txAntennaPort"] + "__" + r.get("txExpanderPort", "NONE")
        seq = TX_ORDER.get(tx_key, 99)
        if prev_seq is not None and seq < prev_seq:
            antRound += 1
        round_to_ar[r["round"]] = antRound
        round_to_tx[r["round"]] = tx_key
        prev_seq = seq

        rxc = r.get("rxAntennaConfig", {})
        rx1 = rxc.get("antennaPort1", "NONE") + "__" + rxc.get("expanderPort1", "NONE")
        rx2 = rxc.get("antennaPort2", "NONE") + "__" + rxc.get("expanderPort2", "NONE")
        round_to_rx[r["round"]] = (rx1, rx2)

    return round_to_ar, round_to_tx, round_to_rx


print("Helpers defined.")

## 3. Discover all position-labelled files

In [ ]:
all_files = []
for txt in sorted(RAW_ROOT.rglob("*.txt")):
    # Only include files whose parent pgm is in INCLUDE_PGMS
    pgm = next((p for p in txt.parts if p.startswith("pgm=")), None)
    if pgm not in INCLUDE_PGMS:
        continue
    meta = parse_filename(txt)
    if meta is None:
        continue
    # Extract run_date from parent id=... directory
    run_date = next((p for p in txt.parts if p.startswith("id=")), "unknown")
    meta["path"]     = txt
    meta["pgm"]      = pgm
    meta["run_date"] = run_date
    meta["file_id"]  = f"{pgm}__{run_date}__x={meta['x']}__y={meta['y']}__z={meta['z']}__P={meta['freq_P']}"
    meta["position_id"] = f"x={meta['x']}__y={meta['y']}__z={meta['z']}"
    all_files.append(meta)

files_df = pd.DataFrame(all_files)
print(f"Files found: {len(files_df)}")
print(f"Unique positions: {files_df['position_id'].nunique()}")
print(f"Programs: {files_df['pgm'].value_counts().to_dict()}")
print(f"Power levels: {sorted(files_df['freq_P'].unique())}")
files_df.head(3)

## 4. Load tag polarisation from Actuals

In [ ]:
actuals = pd.read_csv(ACTUALS_CSV, sep=";", decimal=",")
# EPC is in column 'EPC'; polar in column 'polar'
epc_polar = dict(zip(actuals["EPC"], actuals["polar"]))
print(f"Tags in actuals: {len(epc_polar)}, polar values: {set(epc_polar.values())}")
actuals.head(3)

## 5. Process one file → long-format events then pivot to per-antRound features

### Step 1 (long events): "melt" each TagReadData into two rows
Each event carries `rssi[0]` at `rx1` and `rssi[1]` at `rx2`.  
Following the original RTLS_SF_1tower notebook, we unpivot into:
- `(Tx, Rx=rx1, rssi=rssi1_dBm)`
- `(Tx, Rx=rx2, rssi=rssi2_dBm)`

Also compute `Delta_rssi = rssi1 - rssi2` (differential between the two Rx antennas for the same round).

### Step 2 (wide pivot): for each (antRound, Tx, Rx) cell
Aggregate max/min/n across all reads and EPCs in that cell.

Aggregating across EPCs is intentional: all tags in the file are at the same ground-truth position.

In [ ]:
RSSI_SENTINEL = -100.0  # values <= this mean "not read"

def file_to_long_events(meta: dict) -> pd.DataFrame | None:
    """Parse one raw file into a long-format events DataFrame.

    Returns one row per (event, Rx channel) = "melted" format.
    Columns: antRound_rel, t_rel_s, Tx, Rx, rssi_dBm, Delta_rssi_dBm
    """
    path: Path = meta["path"]
    tags, rounds = parse_raw_file(path)
    if not tags or not rounds:
        return None

    round_to_ar, round_to_tx, round_to_rx = build_round_info(rounds)

    # Time filter
    t0 = min(e["timestamp"] for e in tags)
    if MAX_DURATION_S > 0:
        tags = [t for t in tags if (t["timestamp"] - t0) / 1000 <= MAX_DURATION_S]
    if not tags:
        return None

    # antRound → start timestamp (for t_rel_s)
    ar_t0: dict[int, float] = {}
    for r in sorted(rounds, key=lambda x: x["timestamp"]):
        ar = round_to_ar.get(r["round"])
        if ar is not None and ar not in ar_t0:
            ar_t0[ar] = r["timestamp"]

    # Melt each TagReadData into 2 rows: one per Rx channel
    rows = []
    for tag in tags:
        rid = tag.get("round")
        if rid not in round_to_ar:
            continue
        ar = round_to_ar[rid]
        tx = round_to_tx[rid]
        rx1, rx2 = round_to_rx[rid]
        rssi1, rssi2 = tag["rssi"][0], tag["rssi"][1]
        delta = rssi1 - rssi2 if rssi1 > RSSI_SENTINEL and rssi2 > RSSI_SENTINEL else np.nan
        t_rel = (ar_t0.get(ar, t0) - t0) / 1000

        # Row for rx1 (rssi1)
        if rssi1 > RSSI_SENTINEL:
            rows.append({
                "antRound_rel": ar, "t_rel_s": t_rel,
                "Tx": tx, "Rx": rx1,
                "rssi_dBm": rssi1, "Delta_rssi_dBm": delta,
            })
        # Row for rx2 (rssi2)
        if rssi2 > RSSI_SENTINEL:
            rows.append({
                "antRound_rel": ar, "t_rel_s": t_rel,
                "Tx": tx, "Rx": rx2,
                "rssi_dBm": rssi2, "Delta_rssi_dBm": delta,
            })

    if not rows:
        return None
    return pd.DataFrame(rows)


def long_to_wide(events: pd.DataFrame, meta: dict) -> pd.DataFrame:
    """Pivot long events into one row per antRound with (Tx, Rx) feature columns.

    For each (antRound, Tx, Rx) cell: compute max, min, count of rssi_dBm.
    For each (antRound, Tx): compute max Delta_rssi_dBm (differential between rx1 and rx2).
    """
    # Aggregate RSSI per (antRound, Tx, Rx)
    grouped = events.groupby(["antRound_rel", "Tx", "Rx"])["rssi_dBm"]
    agg = grouped.agg(["max", "min", "count"]).reset_index()

    # Pivot: one row per antRound, columns = {Tx}_{Rx}_{stat}
    agg["col_max"] = agg["Tx"] + "→" + agg["Rx"] + "_max"
    agg["col_min"] = agg["Tx"] + "→" + agg["Rx"] + "_min"
    agg["col_n"]   = agg["Tx"] + "→" + agg["Rx"] + "_n"

    all_ars = sorted(events["antRound_rel"].unique())
    t_rel_map = events.groupby("antRound_rel")["t_rel_s"].first().to_dict()

    rows = []
    for ar in all_ars:
        row = {
            "file_id":      meta["file_id"],
            "position_id":  meta["position_id"],
            "pgm":          meta["pgm"],
            "run_date":     meta["run_date"],
            "antRound_rel": ar,
            "t_rel_s":      t_rel_map.get(ar, 0.0),
            "x": meta["x"], "y": meta["y"], "z": meta["z"],
            "freq_P":       meta["freq_P"],
            "group_id":     meta["position_id"],
        }
        sub = agg[agg["antRound_rel"] == ar]
        for _, r in sub.iterrows():
            row[r["col_max"]] = r["max"]
            row[r["col_min"]] = r["min"]
            row[r["col_n"]]   = r["count"]
        rows.append(row)

    # Delta features per (antRound, Tx)
    delta_grp = events.dropna(subset=["Delta_rssi_dBm"]).groupby(
        ["antRound_rel", "Tx"])["Delta_rssi_dBm"]
    delta_agg = delta_grp.agg(["max", "min"]).reset_index()

    wide = pd.DataFrame(rows)
    for _, d in delta_agg.iterrows():
        ar, tx = d["antRound_rel"], d["Tx"]
        idx = wide.index[wide["antRound_rel"] == ar]
        if len(idx) > 0:
            wide.loc[idx[0], f"{tx}_Delta_max"] = d["max"]
            wide.loc[idx[0], f"{tx}_Delta_min"] = d["min"]

    return wide


def process_file(meta: dict) -> pd.DataFrame | None:
    """Full pipeline: raw file → wide temporal DataFrame."""
    events = file_to_long_events(meta)
    if events is None or events.empty:
        return None
    return long_to_wide(events, meta)


# Smoke-test on one file
_meta = files_df.iloc[0].to_dict()
_events = file_to_long_events(_meta)
print(f"Long events: {len(_events)} rows")
print(f"Unique (Tx, Rx) pairs: {_events.groupby(['Tx','Rx']).ngroups}")
print(f"Sample rows:")
_events.head(4)

In [ ]:
# Smoke-test wide pivot
_wide = process_file(_meta)
print(f"Wide: {_wide.shape[0]} antRounds × {_wide.shape[1]} columns")
feat_cols = [c for c in _wide.columns if "→" in c or "Delta" in c]
print(f"Feature columns: {len(feat_cols)}")
_wide.head(3)

## 6. Process all files

In [ ]:
from tqdm.auto import tqdm

all_parts = []
skipped = []

for _, row in tqdm(files_df.iterrows(), total=len(files_df), desc="Processing files"):
    meta = row.to_dict()
    df = process_file(meta)
    if df is None or df.empty:
        skipped.append(meta["file_id"])
        continue
    all_parts.append(df)

print(f"\nProcessed: {len(all_parts)} files, skipped: {len(skipped)}")
if skipped:
    print("Skipped:", skipped[:5])

In [ ]:
temporal_ds = pd.concat(all_parts, ignore_index=True)

print(f"Total rows: {len(temporal_ds):,}")
print(f"Unique positions (group_id): {temporal_ds['group_id'].nunique()}")
print(f"Unique files: {temporal_ds['file_id'].nunique()}")
print(f"antRounds per file — min: {temporal_ds.groupby('file_id').size().min()}, "
      f"median: {temporal_ds.groupby('file_id').size().median():.0f}, "
      f"max: {temporal_ds.groupby('file_id').size().max()}")
print(f"Columns: {len(temporal_ds.columns)}")
temporal_ds.head(3)

## 7. Inspect sparsity and temporal structure

In [ ]:
feature_cols = sorted([c for c in temporal_ds.columns if "→" in c and c.endswith("_max")])
print(f"(Tx→Rx)_max feature columns ({len(feature_cols)}):")
for col in feature_cols:
    pct = temporal_ds[col].notna().mean() * 100
    print(f"  {col:55s}  {pct:5.1f}% present")

delta_cols = sorted([c for c in temporal_ds.columns if "Delta" in c])
print(f"\nDelta columns ({len(delta_cols)}):")
for col in delta_cols:
    pct = temporal_ds[col].notna().mean() * 100
    print(f"  {col:55s}  {pct:5.1f}% present")

## 8. Save temporal dataset

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Sequence length distribution
seq_lengths = temporal_ds.groupby("file_id").size()
axes[0].hist(seq_lengths, bins=30, edgecolor="k")
axes[0].set_xlabel("antRounds per file")
axes[0].set_ylabel("count")
axes[0].set_title("Sequence length distribution")

# RSSI evolution for one file — pick a few (Tx→Rx) columns
sample_fid = temporal_ds["file_id"].iloc[0]
sample = temporal_ds[temporal_ds["file_id"] == sample_fid].sort_values("antRound_rel")
plotted = 0
for col in feature_cols:
    if sample[col].notna().sum() > 5 and plotted < 6:
        label = col.replace("__NONE", "").replace("_max", "")
        axes[1].plot(sample["antRound_rel"], sample[col], alpha=0.7, label=label)
        plotted += 1

axes[1].set_xlabel("antRound_rel")
axes[1].set_ylabel("max RSSI (dBm)")
axes[1].set_title(f"RSSI evolution — {sample_fid[:60]}")
axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig("../experiments/results/temporal_dataset_preview.png", dpi=120)
plt.show()

out_temporal = OUT_DIR / "ds_temporal_antRound.csv"
temporal_ds.to_csv(out_temporal, index=False)
print(f"Saved temporal → {out_temporal}  ({temporal_ds.shape})")

In [ ]:
out_temporal = OUT_DIR / "ds_temporal_antRound.csv"
temporal_ds.to_csv(out_temporal, index=False)
print(f"Saved temporal → {out_temporal}  ({temporal_ds.shape})")

## 9. Model-ready representations

Three views of the same data, for different model families.

In [ ]:
# --- View B: Wide temporal (one row = one antRound) ---
# Already built above. Each row is a feature vector for one time step.

all_feature_cols = sorted([c for c in temporal_ds.columns if "→" in c or "Delta" in c])
target_cols = ["x", "y", "z"]

print("=== View B: Wide temporal ===")
print(f"  Feature columns: {len(all_feature_cols)}")
print(f"  Total rows: {len(temporal_ds):,}")
print(f"  Usage: tabular models (ExtraTrees, XGBoost, MLP) treat each antRound row independently")
print()

# --- View B→sequence: pad to fixed-length (T, F) tensor ---
T = int(temporal_ds.groupby("file_id").size().quantile(0.9))
file_ids = temporal_ds["file_id"].unique()
F = len(all_feature_cols)

X_seq = np.full((len(file_ids), T, F), np.nan, dtype=np.float32)
y_xyz = np.zeros((len(file_ids), 3), dtype=np.float32)
mask  = np.zeros((len(file_ids), T), dtype=bool)

for i, fid in enumerate(file_ids):
    chunk = temporal_ds[temporal_ds["file_id"] == fid].sort_values("antRound_rel")
    t_len = min(len(chunk), T)
    X_seq[i, :t_len, :] = chunk[all_feature_cols].values[:t_len].astype(np.float32)
    mask[i, :t_len] = True
    y_xyz[i] = chunk[target_cols].iloc[0].values

print("=== View B → Sequence tensor ===")
print(f"  X_seq shape: {X_seq.shape}   (n_files, T={T}, F={F})")
print(f"  y_xyz shape: {y_xyz.shape}")
print(f"  mask shape:  {mask.shape}")
print(f"  NaN rate within valid timesteps: {np.isnan(X_seq[mask]).mean():.1%}")
print(f"  Usage: LSTM, 1D-CNN, Transformer — feed (T, F) per sample with mask")

In [ ]:
# --- View B → Matrix: reshape each antRound into a (n_Tx, n_Rx) RSSI matrix ---
# Useful for GNNs or attention over antenna pairs.

# Extract unique Tx and Rx from column names
tx_rx_max_cols = [c for c in temporal_ds.columns if "→" in c and c.endswith("_max")]
tx_names = sorted(set(c.split("→")[0] for c in tx_rx_max_cols))
rx_names = sorted(set(c.split("→")[1].replace("_max", "") for c in tx_rx_max_cols))
n_Tx, n_Rx = len(tx_names), len(rx_names)
tx_idx = {t: i for i, t in enumerate(tx_names)}
rx_idx = {r: i for i, r in enumerate(rx_names)}

print(f"=== View B → Matrix ===")
print(f"  Tx antennas ({n_Tx}): {tx_names}")
print(f"  Rx antennas ({n_Rx}): {rx_names}")
print(f"  Matrix shape per antRound: ({n_Tx}, {n_Rx})")
print(f"  Full tensor: (n_files, T, {n_Tx}, {n_Rx})")
print(f"  Usage: GNN over antenna graph, or 2D attention over Tx×Rx grid")

In [ ]:
# Save aggregated view
out_agg = OUT_DIR / "ds_aggregated_TxRx.csv"
aggregated.to_csv(out_agg, index=False)
print(f"Saved aggregated → {out_agg}  ({aggregated.shape})")

In [ ]:
# GroupKFold split example
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
groups = temporal_ds.groupby("file_id")["group_id"].first().loc[file_ids].values

for fold, (train_idx, test_idx) in enumerate(gkf.split(file_ids, groups=groups)):
    print(f"Fold {fold}: train={len(train_idx)} files, test={len(test_idx)} files, "
          f"test positions={len(set(groups[test_idx]))}")